# 01 · Estudiantes: exploración de datos

Proyecto del curso Machine Learning (650064), Universidad de Lima, ciclo 2026-2.  
**Objetivo del notebook:** describir el dataset, revisar faltantes y duplicados, fijar la partición train/test y documentar las trampas que condicionan el modelado.  
**Decisiones:** `documentacion/diseno_experimental.md` (las secciones del diseño se citan como §).

## Autores por sección

| Sección | Autor |
|---|---|
| Todas las secciones de este notebook | [Integrante 4] |
| Funciones de `utilidades.py` | Marcelo Cueto |

## 0. Configuración

**Qué hacemos:** preparamos el entorno y cargamos las librerías.

**Por qué:** la celda 0 hace que el mismo notebook funcione en VS Code y en Colab, con las versiones fijadas en `requirements.txt`.

In [ ]:
# Celda 0. Configuración del entorno: funciona igual en VS Code y en Colab.
import os
import sys

EN_COLAB = "google.colab" in sys.modules
USUARIO_GITHUB = "MCuetoV"
REPOSITORIO = "proyecto-ml-2026-2"

if EN_COLAB:
    # En Colab: clonar el repositorio (o actualizarlo) con el token guardado en Secrets
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not os.path.exists(f"/content/{REPOSITORIO}"):
        !git clone -q https://{token}@github.com/{USUARIO_GITHUB}/{REPOSITORIO}.git /content/{REPOSITORIO}
    os.chdir(f"/content/{REPOSITORIO}")
    !git pull -q
    !pip install -q -r requirements.txt
else:
    # En VS Code: si el notebook se abrió desde notebooks/, subir a la raíz del repositorio
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("Carpeta de trabajo:", os.getcwd())

In [ ]:
# Librerías y funciones comunes del proyecto
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import utilidades

pd.set_option("display.max_columns", 50)
print("Semilla del proyecto:", utilidades.SEMILLA)

## 1. Carga y descripción del dataset

**Qué hacemos:** cargamos los datos y vemos su tamaño, sus variables y la distribución del target.

**Por qué:** la Experimentación del reporte empieza describiendo el dataset (consigna). El archivo usa `;` y una columna trae una tabulación en el nombre (§0).

In [ ]:
# Cargar el CSV original (utilidades solo limpia los nombres de columnas)
datos = utilidades.cargar_estudiantes()
print("Filas y columnas:", datos.shape)
print(datos[utilidades.OBJETIVO_ESTUDIANTES].value_counts())
datos.head()

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

In [ ]:
# Fase 3: tabla de variables (nombre, tipo, grupo, escenario) → utilidades.guardar_tabla(..., "tabla_01_descripcion_estudiantes")

## 2. Valores faltantes y duplicados

**Qué hacemos:** contamos valores faltantes y filas repetidas.

**Por qué:** la consigna pide revisar faltantes; los duplicados harían que una misma fila quede en train y en test (§4.1).

In [ ]:
# Valores faltantes y filas duplicadas en el CSV original
print("Valores faltantes:", datos.isna().sum().sum())
print("Filas duplicadas:", datos.duplicated().sum())

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 3. Partición train/test

**Qué hacemos:** separamos el test de reserva (20%) con la función común.

**Por qué:** todo el análisis que informa decisiones se hace solo con el train (§7.2); el test se usa una sola vez, al final del modelado.

In [ ]:
# Partición train/test única del proyecto (§7): la misma que usa el notebook de modelado
train, test = utilidades.dividir_train_test(datos, "estudiantes")
print("Train:", len(train), "· Test:", len(test))
proporciones = pd.DataFrame({"train": train[utilidades.OBJETIVO_ESTUDIANTES].value_counts(normalize=True),
                             "test": test[utilidades.OBJETIVO_ESTUDIANTES].value_counts(normalize=True)})
proporciones.round(3)

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 4. Análisis univariante y del target (train)

**Qué hacemos:** describimos cada variable por separado y la distribución de las clases.

**Por qué:** el desbalance y los niveles raros definen la métrica y la codificación (§4.2 y §9).

In [ ]:
# Fase 3 (solo train):
# - Distribución del target → figura_02_target_estudiantes
# - Histogramas de las numéricas (notas, edad, macroeconómicas)
# - Niveles de cada variable nominal y cuántos tienen menos del 1% (§0)

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 5. Análisis multivariante y trampas del dataset (train)

**Qué hacemos:** relacionamos variables entre sí y con el target, y verificamos las trampas documentadas.

**Por qué:** las trampas de §3 del análisis (nominales como enteros, fuga temporal, Enrolled, cohorte macroeconómica y desbalance) se deben ver en los datos antes de modelar.

In [ ]:
# Fase 3 (solo train):
# - Variables macroeconómicas: valores distintos y combinaciones (cohorte)
# - Target por carrera, beca, deudor y matrícula al día
# - Ceros en las unidades curriculares del 1.er semestre (¿legítimos?)
# - Correlación entre numéricas y chi² entre nominales y target

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 6. Valores atípicos (diagnóstico)

**Qué hacemos:** aplicamos IQR e Isolation Forest para identificar valores extremos.

**Por qué:** son técnicas del curso; aquí sirven solo como diagnóstico y no se eliminan filas (§4.2).

In [ ]:
# Fase 3 (solo train): IQR por variable numérica e Isolation Forest; contar y describir casos

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 7. Decisiones de preprocesamiento que se derivan

| Hallazgo (sección) | Decisión | Sección del diseño |
|---|---|---|
| _(se completa en la Fase 3)_ | | |

## Guardar resultados en GitHub (solo corrida oficial en Colab)

**Qué hacemos:** subimos a `main` los archivos de `resultados/` que genera este notebook.

**Por qué:** las cifras del reporte deben salir de la misma corrida que deja las salidas visibles en el notebook (§12, regla 4). En VS Code esta celda no hace nada; Marcelo hace el *commit* desde VS Code. Después de esta celda, guardar el notebook con *File → Save a copy in GitHub*.

In [ ]:
# Celda final. Cambiar a True solo en la corrida oficial, después de "Reiniciar y ejecutar todo".
import glob

GUARDAR_EN_GITHUB = False
PATRONES_DE_ESTE_NOTEBOOK = ["resultados/tablas/tabla_01_*estudiantes.csv", "resultados/figuras/figura_0[23]_*estudiantes.png"]
if EN_COLAB and GUARDAR_EN_GITHUB:
    archivos = " ".join(f for patron in PATRONES_DE_ESTE_NOTEBOOK for f in sorted(glob.glob(patron)))
    nombre, correo = userdata.get("GITHUB_NOMBRE"), userdata.get("GITHUB_CORREO")
    !git config user.name "{nombre}"
    !git config user.email "{correo}"
    !git add {archivos}
    !git commit -q -m "01-estudiantes: actualiza resultados de la corrida oficial"
    !git pull -q --rebase
    !git push -q
    print("Archivos subidos:", archivos)